# Dataset creator

### jokes

In [1]:
prompt = "to build a synthetic dataset creator, to create a dataset for the stock exchange data"

In [2]:
import os
import io
import re
import sys
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

/Users/srikanth/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL = "llama3.2"
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [4]:
def chat(message, model_name=None, system_prompt=None):
    model = model_name if model_name else MODEL
    sys_prompt = system_prompt if system_prompt is not None else prompt
    messages = [{"role": "system", "content": sys_prompt}] + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [5]:
chat("to create a dataset for stock exchange nifty 50 for 100 rows kind of excel data, dont give any python code ")

"Here's an overview of how you can create a synthetic dataset for the Nifty 50 index data in Excel, covering 100 rows.\n\n**Dataset Details**\n\n* Market Index: Nifty 50\n* Time Period: 100 rows (approximately 1 year)\n* Data Frequency: Daily\n* Variables:\n\t+ Column A: Date (in mm/dd/yyyy format)\n\t+ Column B: Nifty 50 Index Value\n\t+ Column C: Open Price\n\t+ Column D: High Price\n\t+ Column E: Low Price\n\t+ Column F: Close Price\n\t+ Column G: Volume (in units of 1000)\n\t+ Column H: Total Trade Value (in units of Rs. 10,000)\n\t+ Column I: Change from Previous Close\n\t+ Column J: Percentage Change from Previous Close\n\n**Data Generation Rules**\n\n* For date generation, start from the 1st trading day of a randomly selected year (e.g., 2022) and increment by one day for each subsequent row.\n* Generate random values for the Nifty 50 Index Values and other prices using statistical distributions (e.g., normal or exponential) to simulate real-world market behavior.\n* For volume 

In [6]:
default_prompt = "to build a synthetic dataset creator, to create a dataset for the stock exchange data"

def parse_markdown_table(text):
    """Robust parser for markdown tables, CSV blocks, or structured text into a pandas DataFrame"""
    # 1. Try markdown table pipes
    lines = [l.strip() for l in text.strip().split('\n') if l.strip().startswith('|') and l.strip().endswith('|')]
    if len(lines) >= 2:
        try:
            header = lines[0]
            rows = [l for l in lines[1:] if not all(c in '|-: ' for c in l)]
            if rows:
                clean_rows = []
                for row in [header] + rows:
                    cells = [c.strip().replace('**', '').replace('`', '') for c in row.strip('|').split('|')]
                    clean_rows.append(','.join(['"' + c.replace('"', '""') + '"' for c in cells]))
                df = pd.read_csv(io.StringIO('\n'.join(clean_rows)))
                if not df.empty and df.shape[1] > 1:
                    return df
        except Exception:
            pass

    # 2. Try CSV code blocks ```csv ... ``` or ``` ... ```
    csv_match = re.search(r'```(?:csv)?\s*([\s\S]*?)\s*```', text, re.IGNORECASE)
    if csv_match:
        try:
            df = pd.read_csv(io.StringIO(csv_match.group(1).strip()))
            if not df.empty and df.shape[1] > 1:
                return df
        except Exception:
            pass

    return None

def generate_dataset_response(message, model_name, system_prompt):
    if not message.strip():
        return "⚠️ Please enter a prompt.", pd.DataFrame()
    try:
        raw_output = chat(message, model_name=model_name, system_prompt=system_prompt)
        df = parse_markdown_table(raw_output)
        if df is None or df.empty:
            df = pd.DataFrame({
                "Status": ["No structured table found in the output"],
                "Raw Summary": [raw_output[:250] + "..."] if len(raw_output) > 250 else [raw_output]
            })
        return raw_output, df
    except Exception as e:
        return f"❌ Error: {str(e)}", pd.DataFrame({"Error": [str(e)]})

with gr.Blocks(title="Synthetic Dataset Generator & Viewer - Week 3") as demo:
    gr.Markdown("# 🚀 Synthetic Dataset Generator & Viewer (Week 3 Assignment)")
    
    with gr.Row():
        # Left side: Model dropdown, prompt input, and submit button
        with gr.Column(scale=1):
            model_dropdown = gr.Dropdown(
                choices=["llama3.2", "llama3.1", "mistral", "gpt-4o-mini", "gpt-4o"],
                value="llama3.2",
                label="Model Selection",
                info="Choose the LLM model to use"
            )
            prompt_input = gr.Textbox(
                label="Dataset Requirement / Prompt",
                placeholder="e.g., to create a dataset for stock exchange nifty 50 for 100 rows kind of excel data in markdown table format, dont give any python code",
                lines=5,
                value="to create a dataset for stock exchange nifty 50 for 100 rows kind of excel data in markdown table format, dont give any python code"
            )
            system_prompt_input = gr.Textbox(
                label="System Prompt",
                value=default_prompt,
                lines=2
            )
            with gr.Row():
                submit_btn = gr.Button("Submit / Generate", variant="primary")
                clear_btn = gr.Button("Clear")
                
        # Right side: Side-by-side Dataset Viewer (Interactive Table + Formatted Markdown)
        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.TabItem("📊 Interactive Table Viewer"):
                    dataset_table = gr.DataFrame(
                        label="Parsed Dataset Table",
                        interactive=True,
                        wrap=True
                    )
                with gr.TabItem("📝 Formatted Markdown / Raw Output"):
                    output_display = gr.Markdown(
                        label="Generated Output / Data",
                        value="*The generated dataset will appear here after clicking Submit.*"
                    )

    submit_btn.click(
        fn=generate_dataset_response,
        inputs=[prompt_input, model_dropdown, system_prompt_input],
        outputs=[output_display, dataset_table]
    )

    clear_btn.click(
        fn=lambda: ("", "*Cleared output*", pd.DataFrame()),
        inputs=None,
        outputs=[prompt_input, output_display, dataset_table]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
